### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="hepatitis_c_prediction",
    dataset_year="2018",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5D612",
    download_description="""
We get the data from UCI.

wget https://archive.ics.uci.edu/static/public/571/hcv+data.zip && unzip hcv+data.zip && rm hcv+data.zip && mkdir -p local-data-warehouse/hepatitis_c_prediction && mv hcvdat0.csv local-data-warehouse/hepatitis_c_prediction/
""",
    # References
    academic_reference_bibtex="""@article{hoffmann2018using,
  title={Using machine learning techniques to generate laboratory diagnostic pathways—a case study},
  author={Hoffmann, Georg and Bietenbeck, Andreas and Lichtinghagen, Ralf and Klawonn, Frank},
  journal={Journal of Laboratory and Precision Medicine},
  volume={3},
  number={6},
  year={2018},
  publisher={AME Publishing Company}
}
""",
    academic_reference_bibtex_key="hoffmann2018using",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the data from UCI.

- The hepatitis related classes were defined in the paper, but the blood donor class is not mentioned in it. It seems the paper might be related to a different task than the UCI release in the end. But it has the same features. Thus, after looking more into it, we adjust the classes. We drop the "suspect Blood Donor" class as we cannot map it to being part of a reasonable task. We thus treat the task as predicting whether a patient has one of three versions of hepatitis, or is "just" a blood donor.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Category",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="Category",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "hcvdat0.csv")
print("Loaded data shape:", df.shape)

df = df.drop(columns=["Unnamed: 0"])

df = df[df["Category"] != "0s=suspect Blood Donor"].reset_index(drop=True)

as_cat_type = ["Category", "Sex"]
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (615, 14)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 608
Columns: 13
Use sampling: False (sample size: 608)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['ALP', 'CHE', 'GGT', 'ALT', 'CHOL', 'AST', 'PROT', 'BIL', 'ALB', 'CREA']
Rows remaining as candidates after top-10 filter: 14 (of 608)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Category,Age,Sex,ALB,ALP,ALT,AST,BIL,CHE,CHOL,CREA,GGT,PROT
0,0=Blood Donor,42,m,44.1,46.8,23.8,19.4,7.0,10.83,6.28,95.0,19.7,73.0
1,0=Blood Donor,32,m,44.3,52.3,21.7,22.4,17.2,4.15,3.57,78.0,24.1,75.4
2,0=Blood Donor,48,m,46.4,64.1,29.3,27.6,13.2,10.07,8.28,98.0,28.9,83.3
3,0=Blood Donor,38,m,48.4,44.9,23.4,22.1,7.9,10.53,7.51,87.0,43.2,82.6
4,1=Hepatitis,29,m,49.0,NaN,53.0,39.0,15.0,8.79,3.60,79.0,37.0,90.0


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Category,category,0.0,0.00,4.0,"0=Blood Donor, 3=Cirrhosis, 1=Hepatitis, 2=Fibrosis"
1,Sex,category,0.0,0.00,2.0,"m, f"
2,ALP,float64,18.0,2.96,409.0,"52.5, 61.2, 84.1, 59.5, 59.4, 78.6, 66.3, 61.8, 84.3, 56.0"
3,CHOL,float64,10.0,1.64,309.0,"5.07, 5.1, 5.3, 5.9, 5.73, 5.31, 5.62, 4.43, 4.69, 5.88"
4,ALB,float64,1.0,0.16,183.0,"39.0, 44.7, 39.9, 41.0, 46.4, 43.0, 42.0, 41.2, 43.4, 40.0"
5,ALT,float64,1.0,0.16,336.0,"16.6, 19.9, 18.6, 25.2, 10.2, 17.6, 18.3, 23.0, 17.2, 23.8"
6,PROT,float64,1.0,0.16,193.0,"71.9, 73.1, 69.9, 72.4, 72.0, 70.5, 75.2, 71.8, 71.3, 77.1"
7,AST,float64,0.0,0.00,292.0,"22.0, 23.9, 20.0, 22.1, 17.5, 21.2, 24.7, 19.2, 25.7, 19.4"
8,BIL,float64,0.0,0.00,187.0,"6.0, 7.0, 6.1, 4.1, 5.7, 6.8, 6.3, 6.9, 3.7, 5.8"
9,CHE,float64,0.0,0.00,401.0,"7.52, 9.82, 7.1, 5.95, 7.87, 8.9, 7.5, 6.8, 7.93, 8.84"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Age,608.0,47.291118,9.992705,19.00,77.00
ALB,607.0,41.818781,5.406717,20.00,82.20
ALP,590.0,67.821017,25.274423,11.30,416.60
ALT,607.0,27.601318,21.227539,0.90,258.00
AST,608.0,34.369408,32.622442,12.00,324.00
BIL,608.0,11.474013,19.770558,1.80,254.00
CHE,608.0,8.204885,2.168400,1.42,16.41
CHOL,598.0,5.378829,1.119394,1.43,9.67
CREA,608.0,81.513158,49.720652,8.00,1079.10
GGT,608.0,38.243914,51.953220,4.50,650.90


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column   rank                             
Category 1     0=Blood Donor    533  87.66
         2       3=Cirrhosis     30   4.93
         3       1=Hepatitis     24   3.95
         4        2=Fibrosis     21   3.45
Sex      1                 m    371  61.02
         2                 f    237  38.98

In [8]:
# Target Distribution
target_df

,count,pct
Category,,
0=Blood Donor,533,87.66
3=Cirrhosis,30,4.93
1=Hepatitis,24,3.95
2=Fibrosis,21,3.45


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to hepatitis_c_prediction/019d736b-9eba-79b8-8c5a-78d95163e286


019d736b-9eba-79b8-8c5a-78d95163e286
b48c6bc9ccb541a6f4e340bd9f867f3069e9ca9bdd930a7c31c68b479112db88
